# 3D SDF -> 2D SDF + Adjacency Matrices

This notebook converts 3D SDF files to 2D SDF files and exports one adjacency matrix per molecule.

> If `rdkit` import fails in Jupyter, switch the notebook kernel/interpreter to the same Python environment used in your terminal where `rdkit` is installed.

In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

INPUT_SDF_FILES = [
    "1843_actives_new.sdf",
    "Aromatase_actives_new.sdf",
    "Aromatase_inactives_new.sdf",
]

OUT_ROOT = Path("outputs_sdf2d")
SDF_OUT_DIR = OUT_ROOT / "sdf_2d"
ADJ_OUT_DIR = OUT_ROOT / "adjacency"
META_OUT_DIR = OUT_ROOT / "metadata"

for directory in (SDF_OUT_DIR, ADJ_OUT_DIR, META_OUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Output root:", OUT_ROOT.resolve())


In [ ]:

def load_valid_molecules(sdf_path: Path):
    supplier = Chem.SDMolSupplier(str(sdf_path), removeHs=False, sanitize=True)
    return [(i, m) for i, m in enumerate(supplier) if m is not None]

def get_molecule_name(mol: Chem.Mol, fallback_index: int) -> str:
    if mol.HasProp("_Name") and mol.GetProp("_Name").strip():
        return mol.GetProp("_Name").strip()
    if mol.HasProp("PUBCHEM_COMPOUND_CID"):
        return f"CID_{mol.GetProp('PUBCHEM_COMPOUND_CID')}"
    return f"mol_{fallback_index:06d}"

def to_2d_mol(mol: Chem.Mol) -> Chem.Mol:
    m = Chem.Mol(mol)
    m.RemoveAllConformers()
    AllChem.Compute2DCoords(m)
    return m

def adjacency_binary(mol: Chem.Mol) -> np.ndarray:
    n = mol.GetNumAtoms()
    mat = np.zeros((n, n), dtype=np.uint8)
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        mat[i, j] = 1
        mat[j, i] = 1
    return mat


In [ ]:

summaries = []

for sdf_name in INPUT_SDF_FILES:
    sdf_path = Path(sdf_name)
    if not sdf_path.exists():
        print(f"[SKIP] Missing file: {sdf_path}")
        continue

    mols = load_valid_molecules(sdf_path)
    if not mols:
        print(f"[SKIP] No valid molecules in: {sdf_path}")
        continue

    stem = sdf_path.stem
    out_sdf = SDF_OUT_DIR / f"{stem}_2d.sdf"
    out_adj = ADJ_OUT_DIR / f"{stem}_adjacency.npy"
    out_meta = META_OUT_DIR / f"{stem}_metadata.csv"

    adjacency_mats = []
    records = []
    max_abs_z = 0.0

    writer = Chem.SDWriter(str(out_sdf))
    for idx, mol in mols:
        mol2d = to_2d_mol(mol)
        writer.write(mol2d)

        adj = adjacency_binary(mol2d)
        adjacency_mats.append(adj)

        conf = mol2d.GetConformer()
        z_abs = [abs(conf.GetAtomPosition(a).z) for a in range(mol2d.GetNumAtoms())]
        if z_abs:
            max_abs_z = max(max_abs_z, max(z_abs))

        records.append(
            {
                "molecule_index": idx,
                "name": get_molecule_name(mol2d, idx),
                "atoms": mol2d.GetNumAtoms(),
                "bonds": mol2d.GetNumBonds(),
            }
        )
    writer.close()

    np.save(out_adj, np.array(adjacency_mats, dtype=object), allow_pickle=True)
    pd.DataFrame(records).to_csv(out_meta, index=False)

    summaries.append(
        {
            "input": str(sdf_path),
            "molecules": len(mols),
            "output_sdf": str(out_sdf),
            "adjacency_file": str(out_adj),
            "metadata_file": str(out_meta),
            "max_abs_z": max_abs_z,
        }
    )

    print(f"[OK] {sdf_path} -> {out_sdf} ({len(mols)} molecules)")

pd.DataFrame(summaries)
